# Scenario C — QUBO Construction + Classical Baseline (09b)

This notebook turns Scenario C candidates into a **canonical QUBO** and computes a **classical baseline** to compare against QAOA and AWS Braket SV1.

Key goals:
- Load Scenario C candidates (`data/processed/scenario_C_candidates.csv`)
- Define an optimization objective:
  - maximize benefit
  - penalize cost
  - penalize safety risk (if available)
  - optionally reward feasibility
- Add constraints as QUBO penalties:
  - select exactly K trials
  - (optional) sponsor cap (soft penalty)
- Export canonical QUBO contract:
  - `data/qubo_scenarios/scenario_C_qubo.json`
- Compute a classical baseline selection (greedy) and export:
  - best bitstring
  - selected trials CSV
  - summary CSV

Downstream:
- `09c_aws_braket_qaoa_scenario_C_sv1.ipynb` runs pooled QAOA on SV1 and compares performance to this baseline.


In [1]:
# ============================================================
# Cell 1 — Setup: imports, paths, load Scenario C candidates
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# Paths
PROCESSED_DIR = Path("data/processed")
SCEN_DIR = Path("data/scenarios")
QUBO_DIR = Path("data/qubo_scenarios")
RESULTS_DIR = Path("data/results")
FIG_DIR = Path("outputs/figures")

for d in [QUBO_DIR, RESULTS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PATH_CANDIDATES = PROCESSED_DIR / "scenario_C_candidates.csv"
PATH_IDS = SCEN_DIR / "scenario_C_trial_ids.csv"

if not PATH_CANDIDATES.exists():
    raise FileNotFoundError(f"Missing Scenario C candidates: {PATH_CANDIDATES}")
if not PATH_IDS.exists():
    raise FileNotFoundError(f"Missing Scenario C trial IDs: {PATH_IDS}")

candidates = pd.read_csv(PATH_CANDIDATES)
ids = pd.read_csv(PATH_IDS)["nct_id"].astype(str).tolist()

# Keep only Scenario C IDs (defensive)
candidates["nct_id"] = candidates["nct_id"].astype(str)
candidates = candidates[candidates["nct_id"].isin(ids)].copy().reset_index(drop=True)

print("Loaded:", PATH_CANDIDATES, "shape:", candidates.shape)
display(candidates.head(5))
print("\nColumns:", list(candidates.columns))


Loaded: data/processed/scenario_C_candidates.csv shape: (60, 15)


,nct_id,lead_sponsor_norm,region_label,benefit_score,estimated_trial_cost,enrollment_feasibility_score,brief_title,overall_status,phase,_benefit_raw,_cost_raw,_safety_raw,_pool_score,_rank_score,_sponsor_count
0,NCT06744504,Institute of Hematology & Blood Diseases Hospi...,UNKNOWN,1.0,2.0,1.0,Standard-dose vs Intermediate-dose Cytarabine ...,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893,0.202347,1
1,NCT06713616,Yale University,UNKNOWN,1.0,2.0,1.0,PCORI Comparative Effectiveness Study-Esketami...,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893,0.202347,1
2,NCT06691893,Massachusetts General Hospital,UNKNOWN,1.0,2.0,1.0,Evaluating the Efficacy of RELiZORB in Managin...,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893,0.202347,1
3,NCT06693674,Mayo Clinic,UNKNOWN,1.0,2.0,1.0,Effect of Sacubitril-Valsartan on Cardiac Stru...,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893,0.202347,1
4,NCT05582265,Sun Yat-Sen Memorial Hospital of Sun Yat-Sen U...,UNKNOWN,1.0,2.0,1.0,Tislelizumab Combined With Chemotherapy Follow...,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893,0.202347,1



Columns: ['nct_id', 'lead_sponsor_norm', 'region_label', 'benefit_score', 'estimated_trial_cost', 'enrollment_feasibility_score', 'brief_title', 'overall_status', 'phase', '_benefit_raw', '_cost_raw', '_safety_raw', '_pool_score', '_rank_score', '_sponsor_count']


### What Cell 1 Just Did

- Loaded the Scenario C candidate table and the Scenario C trial ID list.
- Defensive-filtered the candidate table to ensure it contains only Scenario C IDs.
- Printed a preview and full column list so we can decide which objective components are available.


In [3]:
# ============================================================
# Cell 2 — Define Scenario C objective and constraint parameters
# ============================================================

# --- Optimization hyperparameters (portfolio demo defaults) ---
K = 10  # pick exactly K trials

# Objective weights (interpretation):
# - benefit is rewarded (we convert to minimization by using negative benefit)
# - cost is penalized
# - safety risk is penalized (if available)
# - feasibility is rewarded (optional; convert to minimization by subtracting)
LAMBDA_COST = 1.0
LAMBDA_SAFETY = 1.0
LAMBDA_FEAS = 0.5

# Constraint penalty strength (bigger = harder constraint enforcement)
A_K = 50.0

# Optional sponsor-cap soft penalty (discourages >CAP per sponsor, but doesn't hard forbid)
SPONSOR_CAP = 2
A_SPONSOR = 10.0  # soft penalty strength

# Required columns (from 09a)
required = ["nct_id", "lead_sponsor_norm", "benefit_score", "estimated_trial_cost", "enrollment_feasibility_score"]
missing = [c for c in required if c not in candidates.columns]
if missing:
    raise ValueError(f"Missing required Scenario C columns: {missing}")

# Safety signal is optional (we only use it if present)
safety_col = None
for c in ["_safety_raw", "safety_score", "sponsor_safety_score", "safety_risk"]:
    if c in candidates.columns:
        safety_col = c
        break

print("K =", K)
print("Objective weights: cost", LAMBDA_COST, "| safety", LAMBDA_SAFETY, "| feas", LAMBDA_FEAS)
print("Penalty A_K =", A_K)
print("Sponsor cap:", SPONSOR_CAP, "A_SPONSOR =", A_SPONSOR)
print("Safety column:", safety_col)


K = 10
Objective weights: cost 1.0 | safety 1.0 | feas 0.5
Penalty A_K = 50.0
Sponsor cap: 2 A_SPONSOR = 10.0
Safety column: _safety_raw


### What Cell 2 Just Did

- Set Scenario C’s primary constraint: **select exactly K trials** (default K=10).
- Defined a clear, explainable objective made of:
  - benefit (reward)
  - cost (penalty)
  - safety risk (penalty, if available)
  - feasibility (reward)
- Defined penalty strengths for:
  - the “exactly K” constraint (hard QUBO penalty)
  - a soft sponsor-cap penalty to discourage sponsor over-concentration
- Detected whether a safety column is available in the Scenario C candidates table.


In [4]:
# ============================================================
# Cell 3 — Build QUBO matrix Q for Scenario C
#   Minimization form: x^T Q x
# ============================================================

df = candidates.copy().reset_index(drop=True)
n = len(df)

# Extract objective components
benefit = pd.to_numeric(df["benefit_score"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
cost = pd.to_numeric(df["estimated_trial_cost"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
feas = pd.to_numeric(df["enrollment_feasibility_score"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

if safety_col is not None:
    safety = pd.to_numeric(df[safety_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
else:
    safety = np.zeros(n, dtype=float)

# Robust scaling so weights behave sanely even if numbers are large
def robust_scale(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) + 1e-9
    z = (x - med) / (1.4826 * mad)
    # squash extremes slightly for stability
    return np.clip(z, -5, 5)

b = robust_scale(benefit)
c = robust_scale(cost)
f = robust_scale(feas)
s = robust_scale(safety)

# Base linear objective (minimization):
# minimize: (+ cost) + (+ safety) + (- benefit) + (- feasibility)
linear = (LAMBDA_COST * c) + (LAMBDA_SAFETY * s) - (1.0 * b) - (LAMBDA_FEAS * f)

# Initialize Q with linear terms on diagonal
Q = np.zeros((n, n), dtype=float)
np.fill_diagonal(Q, linear)

# --- Exactly-K constraint: A_K * (sum x_i - K)^2 ---
# expands to: A_K * [ sum_i x_i + 2*sum_{i<j} x_i x_j - 2K*sum_i x_i + K^2 ]
# => diagonal add: A_K * (1 - 2K)
# => off-diagonal add: A_K * 2 for i<j
diag_add = A_K * (1.0 - 2.0 * K)
Q[np.diag_indices(n)] += diag_add

# off-diagonal
for i in range(n):
    for j in range(i + 1, n):
        Q[i, j] += 2.0 * A_K

# --- Soft sponsor-cap penalty: discourage selecting > CAP per sponsor ---
# We add pairwise penalty within same sponsor:
# For each sponsor group, add A_SPONSOR to each pair (i<j) in that sponsor group.
# This makes selecting multiple from same sponsor incur extra quadratic cost.
if "lead_sponsor_norm" in df.columns and A_SPONSOR > 0:
    groups = df.groupby("lead_sponsor_norm").indices
    for sponsor, idxs in groups.items():
        if len(idxs) <= SPONSOR_CAP:
            continue
        idxs = list(idxs)
        for a in range(len(idxs)):
            for b2 in range(a + 1, len(idxs)):
                i, j = idxs[a], idxs[b2]
                Q[i, j] += A_SPONSOR

# Symmetrize (store as upper-triangular convention; keep full matrix symmetric to be safe)
Q = np.triu(Q) + np.triu(Q, 1).T

print("Built QUBO matrix Q with shape:", Q.shape)
print("Q stats: min", float(Q.min()), "max", float(Q.max()), "mean", float(Q.mean()))


Built QUBO matrix Q with shape: (60, 60)
Q stats: min -950.0 max 110.0 mean 83.05


### What Cell 3 Just Did

- Converted Scenario C’s objective into a **QUBO minimization** form `xᵀQx`.
- Robustly scaled benefit/cost/feasibility/safety to keep weights stable across different numeric ranges.
- Added a hard “exactly K selections” constraint using a quadratic penalty: `A_K * (Σx - K)^2`.
- Added an optional soft sponsor-cap penalty that discourages selecting too many trials from the same sponsor (pairwise penalties within sponsor groups).
- Produced a symmetric QUBO matrix `Q` ready to export using the canonical contract.


In [5]:
# ============================================================
# Cell 4 — Export canonical QUBO contract for Scenario C
# ============================================================

PATH_QUBO_C = QUBO_DIR / "scenario_C_qubo.json"

payload = {
    "Q": Q.tolist(),
    "nct_ids": df["nct_id"].astype(str).tolist(),
    "meta": {
        "scenario": "C",
        "K": int(K),
        "lambda_cost": float(LAMBDA_COST),
        "lambda_safety": float(LAMBDA_SAFETY),
        "lambda_feas": float(LAMBDA_FEAS),
        "A_K": float(A_K),
        "sponsor_cap": int(SPONSOR_CAP),
        "A_sponsor": float(A_SPONSOR),
        "safety_col": safety_col,
        "n_candidates": int(n),
        "source_candidates_csv": str(PATH_CANDIDATES),
    }
}

# Minimal contract check inline (works even if you didn't import the util notebook here)
if not isinstance(payload.get("Q"), list) or not isinstance(payload.get("nct_ids"), list) or not isinstance(payload.get("meta"), dict):
    raise ValueError("Bad QUBO payload structure; expected keys: Q(list), nct_ids(list), meta(dict).")
if len(payload["nct_ids"]) != len(payload["Q"]):
    raise ValueError("len(nct_ids) must match Q dimension.")

PATH_QUBO_C.write_text(json.dumps(payload, indent=2))
print("Wrote canonical QUBO:", PATH_QUBO_C)
print("n =", len(payload["nct_ids"]))


Wrote canonical QUBO: data/qubo_scenarios/scenario_C_qubo.json
n = 60


### What Cell 4 Just Did

- Exported Scenario C as a canonical QUBO file:
  - `data/qubo_scenarios/scenario_C_qubo.json`
- Stored:
  - the full QUBO matrix `Q`
  - an aligned `nct_ids` variable mapping
  - a `meta` dictionary documenting weights, penalties, and scenario parameters
- This file is now the single source of truth for downstream Braket SV1 QAOA runs.


In [6]:
# ============================================================
# Cell 5 — Classical baseline: greedy select K by objective proxy + constraint-aware tweak
# ============================================================

# Greedy heuristic:
# - score each item by linear term (diagonal) only as a fast proxy
# - pick best while discouraging sponsor concentration
diag = np.diag(Q).copy()

# Lower is better (minimization)
order = np.argsort(diag)

selected = []
sponsor_counts = {}

for idx in order:
    sponsor = df.loc[idx, "lead_sponsor_norm"]
    # Soft sponsor cap: allow up to SPONSOR_CAP before we start skipping heavily
    if sponsor_counts.get(sponsor, 0) >= SPONSOR_CAP:
        continue
    selected.append(idx)
    sponsor_counts[sponsor] = sponsor_counts.get(sponsor, 0) + 1
    if len(selected) >= K:
        break

# If we couldn't fill K due to sponsor cap, fill remaining purely by diag
if len(selected) < K:
    for idx in order:
        if idx in selected:
            continue
        selected.append(idx)
        if len(selected) >= K:
            break

x = np.zeros(n, dtype=int)
x[selected] = 1

E = float(x.T @ Q @ x)
bitstring = "".join([str(int(v)) for v in x.tolist()])

OUT_SELECTED = RESULTS_DIR / "09b_scenario_C_classical_selected_trials.csv"
OUT_SUMMARY  = RESULTS_DIR / "09b_scenario_C_classical_summary.csv"
OUT_BITS     = RESULTS_DIR / "09b_scenario_C_classical_best_bitstring.txt"

df_sel = df.loc[selected].copy()
df_sel["selected"] = 1
df_sel.to_csv(OUT_SELECTED, index=False)

summary = pd.DataFrame([{
    "scenario": "C",
    "n_candidates": n,
    "K": K,
    "objective_energy": E,
    "selected_n": int(x.sum()),
    "unique_sponsors": int(df_sel["lead_sponsor_norm"].nunique()),
    "safety_col": safety_col,
    "qubo_path": str(PATH_QUBO_C),
}])
summary.to_csv(OUT_SUMMARY, index=False)

OUT_BITS.write_text(bitstring)

print("Classical baseline objective energy:", E)
print("Selected_n:", int(x.sum()))
print("Unique sponsors:", int(df_sel["lead_sponsor_norm"].nunique()))
print("Wrote:")
print("  -", OUT_SELECTED)
print("  -", OUT_SUMMARY)
print("  -", OUT_BITS)

display(df_sel[["nct_id","lead_sponsor_norm","region_label","benefit_score","estimated_trial_cost","enrollment_feasibility_score"]].head(10))
display(summary)


Classical baseline objective energy: -420.0
Selected_n: 10
Unique sponsors: 6
Wrote:
  - data/results/09b_scenario_C_classical_selected_trials.csv
  - data/results/09b_scenario_C_classical_summary.csv
  - data/results/09b_scenario_C_classical_best_bitstring.txt


,nct_id,lead_sponsor_norm,region_label,benefit_score,estimated_trial_cost,enrollment_feasibility_score
0,NCT06744504,Institute of Hematology & Blood Diseases Hospi...,UNKNOWN,1.0,2.0,1.0
32,NCT05597540,Assistance Publique - Hôpitaux de Paris,UNKNOWN,1.0,2.0,1.0
33,NCT06742723,AstraZeneca,UNKNOWN,1.0,2.0,1.0
34,NCT05689879,Assistance Publique - Hôpitaux de Paris,UNKNOWN,1.0,2.0,1.0
35,NCT05692180,AstraZeneca,UNKNOWN,1.0,2.0,1.0
39,NCT06698796,Pfizer,UNKNOWN,1.0,2.0,1.0
40,NCT05611801,Pfizer,UNKNOWN,1.0,2.0,1.0
46,NCT06726421,Sun Yat-sen University,UNKNOWN,1.0,2.0,1.0
47,NCT06734702,Sun Yat-sen University,UNKNOWN,1.0,2.0,1.0
52,NCT05795140,Novartis Pharmaceuticals,UNKNOWN,1.0,2.0,1.0


,scenario,n_candidates,K,objective_energy,selected_n,unique_sponsors,safety_col,qubo_path
0,C,60,10,-420.0,10,6,_safety_raw,data/qubo_scenarios/scenario_C_qubo.json


### What Cell 5 Just Did

- Computed a **classical baseline** selection using a greedy heuristic:
  - Sort by the QUBO diagonal (a fast proxy for item attractiveness under the minimization objective).
  - Select K items while trying to avoid sponsor over-concentration (soft sponsor cap heuristic).
- Evaluated the full QUBO energy of the selected set.
- Exported baseline artifacts used for comparisons:
  - selected trials CSV
  - summary CSV
  - best bitstring text file


## Summary

We built Scenario C’s optimization instance and classical reference point:

- Constructed a symmetric QUBO matrix `Q` encoding:
  - objective components (benefit/cost/feasibility/safety where available)
  - a hard “exactly K” constraint via `A_K * (Σx - K)^2`
  - an optional sponsor diversification soft penalty
- Exported the canonical QUBO contract:
  - `data/qubo_scenarios/scenario_C_qubo.json`
- Computed a classical baseline selection (greedy) and exported:
  - `data/results/09b_scenario_C_classical_selected_trials.csv`
  - `data/results/09b_scenario_C_classical_summary.csv`
  - `data/results/09b_scenario_C_classical_best_bitstring.txt`

Next notebook:
- `09c_aws_braket_qaoa_scenario_C_sv1.ipynb` will run pooled QAOA on AWS Braket SV1 using `scenario_C_qubo.json`, export SV1 results to S3 + local artifacts, and compare SV1 performance against this classical baseline.
